# Production RAG: semantic cache, cost guardrails, regression eval suite

The RAG bot from the last few labs is going to production tomorrow. The CTO sat in on the architecture review and asked for three things before launch. A per-query cost guardrail that aborts any retrieval whose estimated cost is above 5 cents (the team got burned last quarter paying Cohere four hundred bucks for a runaway batch rerank). A semantic cache that hits at least 30 percent of repeat queries. A regression eval suite that blocks the deploy when recall at five drops more than three points relative to the last release. You have 120 minutes.

The killer moment: you sabotage your own system prompt mid-lab (one cell deliberately mutates the prompt that the bot has been using all session). You re-run the eval suite. The regression detector reads the two most recent runs, sees the drop, and returns `should_block_deploy=true`. You watched your own guardrail save a bad deploy. That is what production AI engineering feels like.

Estimated time: 120 minutes (multiple eval runs plus cache population plus regression detection). Session window: 150 minutes. Cost: about three bucks on a clean run, four if you fight the regression detector. OpenSearch Serverless is the biggest line at $0.96 per OCU-hour times four OCUs over two hours. Claude Sonnet 4.6 would be second largest except Anthropic prompt caching takes it down ninety percent. Cohere rerank is the third line. Everything else (Upstash, Phoenix, Supabase, OpenAI embeddings) sits on free tier or near-zero.

This is the demo lab for the AI/ML Engineering track. The pitch line: build the RAG system every team is shipping to production in 2026 and almost nobody is doing right. Every checkpoint produces an artifact you can show in an interview: the cost guardrail policy in code, the cache hit-rate trace, the regression eval suite that caught the sabotage. Cleanup is critical and the two-hour auto-destroy Lambda is your safety net.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks==0.7.0 is the track standard. Boto3 talks
# to OpenSearch Serverless and the watchdog Lambda. opensearch-py uses
# requests_aws4auth for SigV4. cohere + openai + anthropic + upstash-redis +
# arize-phoenix + supabase + psycopg2-binary cover the rest of the stack.

!pip install --quiet labrun-checks==0.7.0 boto3==1.35.49 opensearch-py==2.7.1 requests-aws4auth==1.3.1 openai==1.54.4 cohere==5.13.3 anthropic==0.42.0 supabase==2.10.0 psycopg2-binary==2.9.10 upstash-redis==1.2.0 arize-phoenix==5.10.0 tenacity==9.0.0

In [ ]:
# Imports, lab identity, resource names. Postgres table names use underscores
# under the labrun_ prefix so the orphan scan can match by LIKE without
# quoting. AWS resource names use hyphens and the labrun- prefix so the tag
# scan finds them.

import atexit
import getpass
import hashlib
import json
import math
import os
import random
import re
import sys
import time
import uuid
from datetime import datetime, timezone
from typing import Any

import boto3
import psycopg2
from tenacity import retry, stop_after_attempt, wait_exponential

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "production-rag-cache-eval"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

RESOURCE_PREFIX = "labrun-production-rag-cache-eval"
COLLECTION_NAME = f"{RESOURCE_PREFIX}-coll"
ENC_POLICY_NAME = f"{RESOURCE_PREFIX}-enc"
NET_POLICY_NAME = f"{RESOURCE_PREFIX}-net"
ACCESS_POLICY_NAME = f"{RESOURCE_PREFIX}-access"
INDEX_NAME = "chunks"
LAMBDA_NAME = f"{RESOURCE_PREFIX}-cleanup-fn"
LAMBDA_ROLE_NAME = f"{RESOURCE_PREFIX}-cleanup-role"
EVENT_RULE_NAME = f"{RESOURCE_PREFIX}-cleanup-rule"

TABLE_PREFIX = "labrun_production_rag_cache_eval_"
CHUNKS_TABLE = TABLE_PREFIX + "chunks"
EVAL_RESULTS_TABLE = TABLE_PREFIX + "eval_results"
REGRESSION_SUMMARY_TABLE = TABLE_PREFIX + "regression_summary"

UPSTASH_CACHE_PREFIX = "labrun:prodrag:cache:"
UPSTASH_HIT_COUNTER_KEY = "labrun:prodrag:hits"
UPSTASH_MISS_COUNTER_KEY = "labrun:prodrag:misses"

PHOENIX_PROJECT_FALLBACK = "labrun-production-rag-cache-eval"

OPENAI_EMBED_MODEL = "text-embedding-3-small"
OPENAI_EMBED_DIMS = 1536
ANTHROPIC_MODEL = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU_MODEL = "claude-haiku-4-5-20250930"
COHERE_RERANK_MODEL = "rerank-3.5"

CACHE_SIMILARITY_THRESHOLD = 0.95
COST_BUDGET_CENTS = 5
REGRESSION_DELTA_THRESHOLD = 0.03  # 3 percentage points

RUN_ID_PREFIX = uuid.uuid4().hex[:8]

# Deterministic seed so the synthetic corpus + eval set are reproducible.
random.seed(42)


In [ ]:
# NBVAL_SKIP
# BYOK setup. Session token first, then the nine provider keys. Each key gets
# a cheap ping before we touch any paid surface. Phoenix wins if both Phoenix
# and Langfuse keys are set; Langfuse is the fallback.

import anthropic
import cohere
import requests
from openai import OpenAI
from supabase import create_client
from upstash_redis import Redis as UpstashRedis

session_token = getpass.getpass("Paste your session token from labrun.dev: ")

OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
COHERE_API_KEY = getpass.getpass("COHERE_API_KEY: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
UPSTASH_REDIS_REST_URL = getpass.getpass("UPSTASH_REDIS_REST_URL: ").strip()
UPSTASH_REDIS_REST_TOKEN = getpass.getpass("UPSTASH_REDIS_REST_TOKEN: ").strip()
PHOENIX_API_KEY = getpass.getpass("PHOENIX_API_KEY (leave blank to use Langfuse): ").strip()
PHOENIX_PROJECT_NAME = getpass.getpass("PHOENIX_PROJECT_NAME (leave blank to default): ").strip() or PHOENIX_PROJECT_FALLBACK
LANGFUSE_PUBLIC_KEY = ""
LANGFUSE_SECRET_KEY = ""
if not PHOENIX_API_KEY:
    LANGFUSE_PUBLIC_KEY = getpass.getpass("LANGFUSE_PUBLIC_KEY: ").strip()
    LANGFUSE_SECRET_KEY = getpass.getpass("LANGFUSE_SECRET_KEY: ").strip()

_required = [
    ("OPENAI_API_KEY", OPENAI_API_KEY),
    ("AWS_ACCESS_KEY_ID", AWS_ACCESS_KEY_ID),
    ("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY),
    ("SUPABASE_URL", SUPABASE_URL),
    ("SUPABASE_SERVICE_ROLE_KEY", SUPABASE_SERVICE_ROLE_KEY),
    ("COHERE_API_KEY", COHERE_API_KEY),
    ("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY),
    ("UPSTASH_REDIS_REST_URL", UPSTASH_REDIS_REST_URL),
    ("UPSTASH_REDIS_REST_TOKEN", UPSTASH_REDIS_REST_TOKEN),
]
_missing = [name for name, val in _required if not val]
if _missing:
    print(f"Missing credentials: {_missing}. Re-run this cell.")
    raise SystemExit(1)
if not PHOENIX_API_KEY and not (LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY):
    print("Need either PHOENIX_API_KEY or both LANGFUSE_* keys.")
    raise SystemExit(1)

OBS_PROVIDER = "phoenix" if PHOENIX_API_KEY else "langfuse"
print(f"Observability provider for this session: {OBS_PROVIDER}")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# STS GetCallerIdentity. Validates the AWS keys and surfaces the account id.
try:
    sts = boto3.client("sts", region_name=REGION)
    AWS_IDENTITY = sts.get_caller_identity()
    AWS_ACCOUNT_ID = AWS_IDENTITY["Account"]
    print(f"AWS identity validated. Account: {AWS_ACCOUNT_ID}. Region: {REGION}.")
except Exception as exc:
    print(f"AWS credentials rejected: {exc!r}")
    raise SystemExit(1)

# OpenAI ping. One tiny embedding call: roughly a fraction of a cent.
openai_client = OpenAI(api_key=OPENAI_API_KEY)
try:
    _emb = openai_client.embeddings.create(model=OPENAI_EMBED_MODEL, input=["ping"])
    print(f"OpenAI validated. Embedding dims: {len(_emb.data[0].embedding)}.")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    raise SystemExit(1)

# Cohere ping.
co_client = cohere.ClientV2(api_key=COHERE_API_KEY)
try:
    _ = co_client.models.list()
    print("Cohere validated.")
except Exception as exc:
    print(f"Cohere key rejected: {exc!r}")
    raise SystemExit(1)

# Anthropic ping (Haiku, the cheap model).
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with one word: ok"}],
    )
    print(f"Anthropic validated. Haiku replied: {_ping.content[0].text!r}.")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

# Supabase REST client + direct psycopg2 connection for raw SQL.
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
DB_URI = getpass.getpass(
    "Supabase Postgres connection string "
    "(Project Settings > Database > Connection string > URI; "
    "use the Transaction pooler URL): "
).strip()
try:
    _conn = psycopg2.connect(DB_URI)
    _conn.autocommit = True
    with _conn.cursor() as _cur:
        _cur.execute("SELECT 1")
        _cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    _conn.close()
    print("Supabase Postgres validated. pgvector extension confirmed.")
except Exception as exc:
    print(f"Postgres connection failed: {exc!r}")
    raise SystemExit(1)

# Upstash Redis ping via REST.
upstash = UpstashRedis(url=UPSTASH_REDIS_REST_URL, token=UPSTASH_REDIS_REST_TOKEN)
try:
    pong = upstash.ping()
    print(f"Upstash Redis validated. PING -> {pong!r}.")
except Exception as exc:
    print(f"Upstash credentials rejected: {exc!r}")
    raise SystemExit(1)

# Phoenix or Langfuse project ping. Both surfaces are HTTP; we hit a trivial
# endpoint to confirm credentials work before we start emitting traces.
if OBS_PROVIDER == "phoenix":
    try:
        _r = requests.get(
            "https://app.phoenix.arize.com/v1/projects",
            headers={"Authorization": f"Bearer {PHOENIX_API_KEY}"},
            timeout=10,
        )
        if _r.status_code >= 400:
            raise RuntimeError(f"Phoenix returned {_r.status_code}: {_r.text[:200]}")
        print(f"Phoenix validated. Project: {PHOENIX_PROJECT_NAME!r}.")
    except Exception as exc:
        print(f"Phoenix key rejected: {exc!r}")
        raise SystemExit(1)
else:
    try:
        _r = requests.get(
            "https://cloud.langfuse.com/api/public/projects",
            auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
            timeout=10,
        )
        if _r.status_code >= 400:
            raise RuntimeError(f"Langfuse returned {_r.status_code}: {_r.text[:200]}")
        print("Langfuse validated.")
    except Exception as exc:
        print(f"Langfuse key rejected: {exc!r}")
        raise SystemExit(1)

# Bundle creds for the cleanup adapter to consume.
LAB_CREDENTIALS = {
    "region": REGION,
    "aws_account_id": AWS_ACCOUNT_ID,
    "supabase_url": SUPABASE_URL,
    "service_role_key": SUPABASE_SERVICE_ROLE_KEY,
    "db_uri": DB_URI,
    "upstash_url": UPSTASH_REDIS_REST_URL,
    "upstash_token": UPSTASH_REDIS_REST_TOKEN,
    "phoenix_api_key": PHOENIX_API_KEY,
    "phoenix_project_name": PHOENIX_PROJECT_NAME,
    "langfuse_public_key": LANGFUSE_PUBLIC_KEY,
    "langfuse_secret_key": LANGFUSE_SECRET_KEY,
    "obs_provider": OBS_PROVIDER,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=LAB_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}.")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest is critical-first per RESOURCE-SAFETY-SPEC Section 2 Rule 2.
# OpenSearch Serverless is the only hourly-billed surface (OCU charges) so the
# collection tears down first, then the three security policies. The watchdog
# Lambda + EventBridge rule + IAM role come next so the safety net cannot
# linger past cleanup. Upstash index, Phoenix project, and the three pgvector
# tables follow.
#
# TODO: migrate to observability.py + upstash.py + vector_db.py adapters once
# they ship (see TRACK-BRIEF.md Section 4). For now the adapter is inline so
# the demo lab is self-contained.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoss_collection",
        id=COLLECTION_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=f"aws opensearchserverless delete-collection --id <id> --region {REGION}",
    ),
    CleanupResource(
        type="aoss_security_policy",
        id=ENC_POLICY_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=f"aws opensearchserverless delete-security-policy --name {ENC_POLICY_NAME} --type encryption --region {REGION}",
        extra={"policy_type": "encryption"},
    ),
    CleanupResource(
        type="aoss_security_policy",
        id=NET_POLICY_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=f"aws opensearchserverless delete-security-policy --name {NET_POLICY_NAME} --type network --region {REGION}",
        extra={"policy_type": "network"},
    ),
    CleanupResource(
        type="aoss_access_policy",
        id=ACCESS_POLICY_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=f"aws opensearchserverless delete-access-policy --name {ACCESS_POLICY_NAME} --type data --region {REGION}",
    ),
    CleanupResource(
        type="events_rule",
        id=EVENT_RULE_NAME,
        region=REGION,
        cli_delete_command=f"aws events remove-targets --rule {EVENT_RULE_NAME} --ids 1 && aws events delete-rule --name {EVENT_RULE_NAME} --region {REGION}",
    ),
    CleanupResource(
        type="lambda_function",
        id=LAMBDA_NAME,
        region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {LAMBDA_NAME} --region {REGION}",
    ),
    CleanupResource(
        type="iam_role",
        id=LAMBDA_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {LAMBDA_ROLE_NAME}",
    ),
    CleanupResource(
        type="upstash_keys",
        id=UPSTASH_CACHE_PREFIX,
        region="upstash",
        cli_delete_command="upstash redis flushdb (or scan + del via REST)",
    ),
    CleanupResource(
        type="phoenix_project",
        id=PHOENIX_PROJECT_NAME,
        region="observability",
        cli_delete_command="DELETE /v1/projects/<id> via Phoenix API",
    ),
    CleanupResource(
        type="supabase_table",
        id=REGRESSION_SUMMARY_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{REGRESSION_SUMMARY_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="supabase_table",
        id=EVAL_RESULTS_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{EVAL_RESULTS_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="supabase_table",
        id=CHUNKS_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{CHUNKS_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="local_file",
        id="/content/regression_run_baseline.json",
        region="local",
        cli_delete_command="rm -f /content/regression_run_*.json",
    ),
    CleanupResource(
        type="local_file",
        id="/content/regression_run_sabotaged.json",
        region="local",
        cli_delete_command="rm -f /content/regression_run_*.json",
    ),
]


class ProdRagCleanupAdapter:
    """Inline adapter for OpenSearch Serverless, Supabase, Upstash, Phoenix,
    Lambda, EventBridge, IAM, and local files. Each delete_resource branch
    swallows the canonical 'already gone' error for that surface and re-raises
    everything else so run_cleanup turns it into a warning with the CLI command.

    TODO: migrate to observability.py + upstash.py + vector_db.py adapters once
    those ship. The branches stay structured per resource type so the migration
    is mechanical.
    """

    def __init__(self, credentials):
        self._creds = credentials
        self._region = credentials["region"]
        self._aoss = boto3.client("opensearchserverless", region_name=self._region)
        self._lambda = boto3.client("lambda", region_name=self._region)
        self._events = boto3.client("events", region_name=self._region)
        self._iam = boto3.client("iam", region_name=self._region)
        self._db_uri = credentials["db_uri"]
        self._upstash = UpstashRedis(
            url=credentials["upstash_url"], token=credentials["upstash_token"]
        )

    def _exec_sql(self, sql):
        conn = psycopg2.connect(self._db_uri)
        conn.autocommit = True
        try:
            with conn.cursor() as cur:
                cur.execute(sql)
        finally:
            conn.close()

    def _table_exists(self, name):
        conn = psycopg2.connect(self._db_uri)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT 1 FROM information_schema.tables "
                    "WHERE table_schema = 'public' AND table_name = %s",
                    (name,),
                )
                return cur.fetchone() is not None
        finally:
            conn.close()

    def _collection_id(self, name):
        resp = self._aoss.list_collections(collectionFilters={"name": name})
        for s in resp.get("collectionSummaries", []):
            if s.get("name") == name:
                return s.get("id")
        return None

    def _wait_collection_deleted(self, collection_id, ceiling_seconds=300):
        deadline = time.time() + ceiling_seconds
        while time.time() < deadline:
            try:
                resp = self._aoss.batch_get_collection(ids=[collection_id])
                details = resp.get("collectionDetails", []) or []
                if not details:
                    return True
                state = details[0].get("status")
                if state in (None, "DELETED"):
                    return True
            except self._aoss.exceptions.ResourceNotFoundException:
                return True
            except Exception:
                return True
            time.sleep(15)
        return False

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "aoss_collection":
            cid = self._collection_id(rid)
            if cid is None:
                return
            try:
                self._aoss.delete_collection(id=cid)
            except self._aoss.exceptions.ResourceNotFoundException:
                return
            self._wait_collection_deleted(cid)
            return
        if rtype == "aoss_security_policy":
            ptype = (resource.extra or {}).get("policy_type", "encryption")
            try:
                self._aoss.delete_security_policy(name=rid, type=ptype)
            except self._aoss.exceptions.ResourceNotFoundException:
                return
            return
        if rtype == "aoss_access_policy":
            try:
                self._aoss.delete_access_policy(name=rid, type="data")
            except self._aoss.exceptions.ResourceNotFoundException:
                return
            return
        if rtype == "events_rule":
            try:
                self._events.remove_targets(Rule=rid, Ids=["1"], Force=True)
            except Exception:
                pass
            try:
                self._events.delete_rule(Name=rid, Force=True)
            except self._events.exceptions.ResourceNotFoundException:
                return
            return
        if rtype == "lambda_function":
            try:
                self._lambda.delete_function(FunctionName=rid)
            except self._lambda.exceptions.ResourceNotFoundException:
                return
            return
        if rtype == "iam_role":
            try:
                policies = self._iam.list_role_policies(RoleName=rid).get("PolicyNames", [])
                for pname in policies:
                    self._iam.delete_role_policy(RoleName=rid, PolicyName=pname)
                attached = self._iam.list_attached_role_policies(RoleName=rid).get("AttachedPolicies", [])
                for ap in attached:
                    self._iam.detach_role_policy(RoleName=rid, PolicyArn=ap["PolicyArn"])
                self._iam.delete_role(RoleName=rid)
            except self._iam.exceptions.NoSuchEntityException:
                return
            return
        if rtype == "upstash_keys":
            try:
                cursor = 0
                while True:
                    cursor, batch = self._upstash.scan(cursor, match=f"{rid}*", count=200)
                    if batch:
                        self._upstash.delete(*batch)
                    if cursor == 0:
                        break
                self._upstash.delete(UPSTASH_HIT_COUNTER_KEY, UPSTASH_MISS_COUNTER_KEY)
            except Exception:
                pass
            return
        if rtype == "phoenix_project":
            try:
                if self._creds.get("obs_provider") == "phoenix":
                    requests.delete(
                        f"https://app.phoenix.arize.com/v1/projects/{rid}",
                        headers={"Authorization": f"Bearer {self._creds.get('phoenix_api_key', '')}"},
                        timeout=10,
                    )
            except Exception:
                pass
            return
        if rtype == "supabase_table":
            self._exec_sql(f"DROP TABLE IF EXISTS public.{rid} CASCADE")
            return
        if rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass
            return
        raise ValueError(f"ProdRagCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "aoss_collection":
            return self._collection_id(rid) is not None
        if rtype == "aoss_security_policy":
            ptype = (resource.extra or {}).get("policy_type", "encryption")
            try:
                self._aoss.get_security_policy(name=rid, type=ptype)
                return True
            except self._aoss.exceptions.ResourceNotFoundException:
                return False
            except Exception:
                return False
        if rtype == "aoss_access_policy":
            try:
                self._aoss.get_access_policy(name=rid, type="data")
                return True
            except self._aoss.exceptions.ResourceNotFoundException:
                return False
            except Exception:
                return False
        if rtype == "events_rule":
            try:
                self._events.describe_rule(Name=rid)
                return True
            except self._events.exceptions.ResourceNotFoundException:
                return False
            except Exception:
                return False
        if rtype == "lambda_function":
            try:
                self._lambda.get_function(FunctionName=rid)
                return True
            except self._lambda.exceptions.ResourceNotFoundException:
                return False
            except Exception:
                return False
        if rtype == "iam_role":
            try:
                self._iam.get_role(RoleName=rid)
                return True
            except self._iam.exceptions.NoSuchEntityException:
                return False
            except Exception:
                return False
        if rtype == "upstash_keys":
            try:
                cursor, batch = self._upstash.scan(0, match=f"{rid}*", count=1)
                return len(batch) > 0
            except Exception:
                return False
        if rtype == "phoenix_project":
            return False
        if rtype == "supabase_table":
            try:
                return self._table_exists(rid)
            except Exception:
                return False
        if rtype == "local_file":
            return os.path.exists(rid)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        if region == self._region:
            try:
                rgt = boto3.client("resourcegroupstaggingapi", region_name=self._region)
                paginator = rgt.get_paginator("get_resources")
                for page in paginator.paginate(
                    TagFilters=[{"Key": "labrun:lab-slug", "Values": [lab_slug]}]
                ):
                    for r in page.get("ResourceTagMappingList", []):
                        arns.append(r["ResourceARN"])
            except Exception:
                pass
            return arns
        if region == "supabase":
            try:
                conn = psycopg2.connect(self._db_uri)
                try:
                    with conn.cursor() as cur:
                        cur.execute(
                            "SELECT table_name FROM information_schema.tables "
                            "WHERE table_schema = 'public' AND table_name LIKE %s",
                            (TABLE_PREFIX + "%",),
                        )
                        return [row[0] for row in cur.fetchall()]
                finally:
                    conn.close()
            except Exception:
                return []
        if region == "upstash":
            try:
                cursor, batch = self._upstash.scan(0, match=f"{UPSTASH_CACHE_PREFIX}*", count=200)
                return list(batch or [])
            except Exception:
                return []
        return arns


CLEANUP_ADAPTER = ProdRagCleanupAdapter(LAB_CREDENTIALS)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


# Orphan scan. Per RESOURCE-SAFETY-SPEC Section 2 Rule 4 the scan BLOCKS, not warns.
def scan_for_orphans():
    aws_orphans = CLEANUP_ADAPTER.tag_scan(LAB_CREDENTIALS, LAB_TAG_VALUE, REGION)
    sb_orphans = CLEANUP_ADAPTER.tag_scan(LAB_CREDENTIALS, LAB_TAG_VALUE, "supabase")
    up_orphans = CLEANUP_ADAPTER.tag_scan(LAB_CREDENTIALS, LAB_TAG_VALUE, "upstash")
    return aws_orphans, sb_orphans, up_orphans


aws_o, sb_o, up_o = scan_for_orphans()
if aws_o or sb_o or up_o:
    print("BLOCKED: leftover resources from a prior session detected.")
    for arn in aws_o:
        print("  AWS:", arn)
    for name in sb_o:
        print("  Supabase table:", name)
    for key in up_o:
        print("  Upstash key:", key)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

In [ ]:
# NBVAL_SKIP
# Build the synthetic corpus (about 2 MB of markdown spread across ~240 chunks)
# and the 100-query deterministic eval set (70 originals + 30 paraphrases of
# the first 30 originals). The seed is pinned in the imports cell. The corpus
# stays in-memory; chunks land in OpenSearch + pgvector in Task 1.

TOPICS = [
    "semantic caching", "hybrid retrieval", "reranking", "prompt caching",
    "cost guardrails", "regression eval", "chunk overlap", "BM25 scoring",
    "dense embeddings", "recall at five", "production rag", "observability",
]

SUBTOPICS = [
    "how it works", "common pitfalls", "failure modes", "cost math",
    "tuning levers", "production patterns", "trade-offs", "monitoring",
]

def synth_chunk(topic, subtopic, idx):
    base = (
        f"In production RAG, {topic} is most often misunderstood as a {subtopic}. "
        f"The {idx}-th implementation detail is the threshold tuning: paraphrase "
        f"distributions vary across corpora, and the {topic} component is sensitive "
        f"to that distribution. Teams that ship {topic} without a regression eval "
        f"suite are flying blind. The CTO who got burned twice in six months by "
        f"silent regressions in the system prompt is the canonical case study. "
        f"Cite every chunk you reference when answering questions about {topic}."
    )
    pad = " ".join(["detail-" + str(idx)] * 30)
    return base + " " + pad

CORPUS = []  # list of (chunk_id, text, label_set)
for t_idx, topic in enumerate(TOPICS):
    for s_idx, sub in enumerate(SUBTOPICS):
        for k in range(3):
            chunk_id = f"c-{t_idx:02d}-{s_idx:02d}-{k:02d}"
            text = synth_chunk(topic, sub, t_idx * 100 + s_idx * 10 + k)
            CORPUS.append((chunk_id, text, {"topic": topic, "subtopic": sub}))

print(f"Synthetic corpus: {len(CORPUS)} chunks, ~{sum(len(t) for _, t, _ in CORPUS) // 1024} KB.")

QUERY_TEMPLATES = [
    "How does {topic} interact with {subtopic} in production?",
    "What are the common pitfalls of {topic} at scale?",
    "Explain {topic} and why it matters for {subtopic}.",
    "In production rag, what does the {topic} component do for {subtopic}?",
    "What is the cost math behind {topic}?",
    "How do teams monitor {topic} for {subtopic} regressions?",
    "Give me production patterns for {topic} when handling {subtopic}.",
]

PARAPHRASE_PAIRS = [
    ("How does", "What is the way that"),
    ("What are the", "Can you list the"),
    ("Explain", "Walk me through"),
    ("In production rag, what does", "For a production-grade RAG, what does"),
    ("What is the cost math", "Could you break down the cost"),
    ("How do teams monitor", "What do teams use to keep an eye on"),
    ("Give me production patterns", "Share real-world patterns"),
]

def paraphrase(q):
    out = q
    for orig, repl in PARAPHRASE_PAIRS:
        if out.startswith(orig):
            out = repl + out[len(orig):]
            break
    return out

EVAL_SET = []
random.seed(42)
originals = []
for i in range(70):
    topic = TOPICS[i % len(TOPICS)]
    sub = SUBTOPICS[i % len(SUBTOPICS)]
    template = QUERY_TEMPLATES[i % len(QUERY_TEMPLATES)]
    query = template.format(topic=topic, subtopic=sub)
    gold_ids = [c_id for c_id, _, labels in CORPUS if labels["topic"] == topic]
    EVAL_SET.append({
        "query_id": f"q-{i:03d}",
        "query": query,
        "gold_chunk_ids": gold_ids,
        "is_paraphrase": False,
        "paraphrase_of": None,
    })
    originals.append((i, query, gold_ids))

for i in range(30):
    src_idx, src_query, src_gold = originals[i]
    EVAL_SET.append({
        "query_id": f"q-p-{i:03d}",
        "query": paraphrase(src_query),
        "gold_chunk_ids": src_gold,
        "is_paraphrase": True,
        "paraphrase_of": f"q-{src_idx:03d}",
    })

print(f"Eval set: {len(EVAL_SET)} queries ({sum(1 for q in EVAL_SET if q['is_paraphrase'])} paraphrases).")


## Task 1: Stand up the retrieval substrate

This is the same shape as Lab 02: an OpenSearch Serverless collection holds the BM25 index, and Supabase pgvector holds the dense index. The corpus is already in memory from the seed cell above. You write the indexing logic: create the collection plus three policies, wait for ACTIVE, create the `chunks` index, embed the corpus through OpenAI text-embedding-3-small, and upsert into both surfaces.

Pass criteria: the OpenSearch collection is ACTIVE, the `chunks` index has at least 200 documents, and the `labrun_production_rag_cache_eval_chunks` pgvector table holds the same row count (within a tolerance of 1). The validator queries both surfaces; do not let pgvector and OpenSearch drift.

Cost during this task: about 20 cents of OpenSearch OCU charges over the indexing window (the collection takes 60 to 120 seconds to come ACTIVE), plus less than a penny of OpenAI embeddings for the 240 chunks. The two-hour watchdog Lambda from Cell 4's manifest is already armed in case cleanup never runs.

In [ ]:
# NBVAL_SKIP
# Level 2 skeleton. The OpenSearch Serverless policies, the index mapping, the
# pgvector table, and the boto3 + opensearch-py + supabase client wiring are
# provided. You write the four boto3 / opensearch-py / psycopg2 calls that
# actually create the collection, the index, and the two parallel writes
# (BM25 doc into OpenSearch, dense vector row into pgvector) for each chunk.

from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

# 1) Encryption + network + access policies. AOSS requires all three before
#    the collection itself can be created.
_enc_doc = {
    "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
    "AWSOwnedKey": True,
}
_net_doc = [{
    "Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
        {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]},
    ],
    "AllowFromPublic": True,
}]
_access_doc = [{
    "Rules": [{
        "ResourceType": "index",
        "Resource": [f"index/{COLLECTION_NAME}/*"],
        "Permission": ["aoss:*"],
    }, {
        "ResourceType": "collection",
        "Resource": [f"collection/{COLLECTION_NAME}"],
        "Permission": ["aoss:*"],
    }],
    "Principal": [f"arn:aws:iam::{AWS_ACCOUNT_ID}:root"],
}]

aoss = boto3.client("opensearchserverless", region_name=REGION)

def _create_policy_idempotent(method, name, policy_type, doc):
    try:
        method(name=name, type=policy_type, policy=json.dumps(doc))
    except aoss.exceptions.ConflictException:
        pass

_create_policy_idempotent(aoss.create_security_policy, ENC_POLICY_NAME, "encryption", _enc_doc)
_create_policy_idempotent(aoss.create_security_policy, NET_POLICY_NAME, "network", _net_doc)
_create_policy_idempotent(aoss.create_access_policy, ACCESS_POLICY_NAME, "data", _access_doc)
print("Three AOSS policies created (encryption, network, data access).")

# 2) Create the collection.
# YOUR CODE: call aoss.create_collection(name=COLLECTION_NAME, type='SEARCH',
# tags=[{'key': 'labrun:lab-slug', 'value': LAB_TAG_VALUE}]) and capture the id.
# Swallow ConflictException if the collection already exists; in that case look
# the id up with aoss.list_collections(collectionFilters={'name': COLLECTION_NAME}).
collection_id = None  # YOUR CODE assigns this

# 3) Wait for the collection to be ACTIVE. The status goes CREATING -> ACTIVE
#    over roughly 60 to 120 seconds. Poll batch_get_collection every 15s,
#    ceiling 5 minutes.
def wait_collection_active(cid, ceiling_seconds=300):
    msgs = ["warming the OCUs", "summoning index shards", "convincing AOSS to acknowledge us", "still creating, this always feels longer than 90 seconds"]
    start = time.time()
    i = 0
    while time.time() - start < ceiling_seconds:
        resp = aoss.batch_get_collection(ids=[cid])
        details = resp.get("collectionDetails", [])
        if details and details[0].get("status") == "ACTIVE":
            print(f"Collection ACTIVE after {int(time.time() - start)}s.")
            return details[0]
        print(f"  {msgs[i % len(msgs)]}... ({int(time.time() - start)}s)")
        i += 1
        time.sleep(15)
    raise TimeoutError("Collection did not reach ACTIVE within ceiling")

# YOUR CODE: call wait_collection_active(collection_id) and capture the host
# from the returned dict (look for 'collectionEndpoint').
collection_detail = None  # YOUR CODE assigns this
AOSS_HOST = None           # YOUR CODE assigns from collection_detail['collectionEndpoint']

# 4) Build the SigV4-auth OpenSearch client.
_credentials = boto3.Session().get_credentials()
_awsauth = AWS4Auth(
    _credentials.access_key,
    _credentials.secret_key,
    REGION,
    "aoss",
    session_token=_credentials.token,
)
_host_no_scheme = (AOSS_HOST or "").replace("https://", "").replace("http://", "")
os_client = OpenSearch(
    hosts=[{"host": _host_no_scheme, "port": 443}],
    http_auth=_awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    pool_maxsize=20,
)

# 5) Create the chunks index with a BM25-friendly mapping.
_index_body = {
    "settings": {"index": {"knn": False}},
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "text": {"type": "text"},
            "topic": {"type": "keyword"},
            "subtopic": {"type": "keyword"},
        }
    },
}
# YOUR CODE: call os_client.indices.create(index=INDEX_NAME, body=_index_body)
# and swallow the index_already_exists_exception.

# 6) Create the pgvector table that holds the dense index.
_pgvector_ddl = f"""
CREATE TABLE IF NOT EXISTS public.{CHUNKS_TABLE} (
    chunk_id text PRIMARY KEY,
    text text NOT NULL,
    topic text NOT NULL,
    subtopic text NOT NULL,
    embedding vector({OPENAI_EMBED_DIMS})
);
"""
conn = psycopg2.connect(DB_URI)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(_pgvector_ddl)
conn.close()
print(f"pgvector table {CHUNKS_TABLE} ready.")

# 7) Embed the corpus in batches.
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, max=10))
def embed_batch(texts):
    resp = openai_client.embeddings.create(model=OPENAI_EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

BATCH = 32
embeddings_by_id = {}
for i in range(0, len(CORPUS), BATCH):
    batch = CORPUS[i:i + BATCH]
    vecs = embed_batch([text for _, text, _ in batch])
    for (chunk_id, _, _), vec in zip(batch, vecs):
        embeddings_by_id[chunk_id] = vec
print(f"Embedded {len(embeddings_by_id)} chunks.")

# 8) Bulk-write to both surfaces. For each chunk: index a BM25 doc into
#    OpenSearch, upsert a dense row into pgvector.
# YOUR CODE: write the parallel-write loop. OpenSearch doc shape:
#   {'chunk_id': ..., 'text': ..., 'topic': ..., 'subtopic': ...}
# Use os_client.index(index=INDEX_NAME, body=doc) per chunk.
# pgvector: INSERT INTO {CHUNKS_TABLE} (chunk_id, text, topic, subtopic, embedding)
#   VALUES (%s, %s, %s, %s, %s::vector) ON CONFLICT (chunk_id) DO UPDATE SET ...
chunks_written = 0  # YOUR CODE updates this

print(f"Indexed {chunks_written} chunks into OpenSearch + pgvector.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: collection ACTIVE; index doc count >= 200; pgvector row count
# matches OpenSearch doc count within 1.

def checkpoint_1(session):
    try:
        resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
        details = resp.get("collectionDetails", [])
        if not details:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OpenSearch collection {COLLECTION_NAME!r} not found. "
                    f"Did the create_collection call succeed? Check the AWS console under "
                    f"OpenSearch Serverless."
                ),
            )
        status_str = details[0].get("status")
        if status_str != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Collection status is {status_str!r}, expected 'ACTIVE'. "
                    f"Re-run wait_collection_active(collection_id). AOSS can take up to two minutes."
                ),
            )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"AOSS describe failed: {exc!r}",
        )

    try:
        os_count = os_client.count(index=INDEX_NAME).get("count", 0)
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"OpenSearch count failed: {exc!r}. Did the chunks index get created?",
        )
    if os_count < 200:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"OpenSearch chunks index holds {os_count} docs, expected >= 200. "
                f"Confirm the bulk-write loop iterated over the full CORPUS list "
                f"({len(CORPUS)} chunks)."
            ),
        )

    try:
        conn = psycopg2.connect(DB_URI)
        try:
            with conn.cursor() as cur:
                cur.execute(f"SELECT COUNT(*) FROM public.{CHUNKS_TABLE}")
                pg_count = cur.fetchone()[0]
        finally:
            conn.close()
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"pgvector count failed: {exc!r}",
        )

    if abs(pg_count - os_count) > 1:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"pgvector row count {pg_count} does not match OpenSearch doc count {os_count} "
                f"(tolerance 1). The two surfaces drifted. Most common cause: the bulk-write loop "
                f"committed to one side but skipped a chunk on the other (try/except swallowing a "
                f"single insert failure)."
            ),
        )

    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three things have to land: the collection has to exist and be ACTIVE, the `chunks` index has to be created on the collection, and the bulk-write loop has to write to both surfaces in the same pass. If the validator says the collection is missing, the create_collection call did not succeed. If the validator says counts drifted, one side's insert failed silently.

</details>

<details><summary>Hint 2 (direction)</summary>

For the collection:
```
resp = aoss.create_collection(
    name=COLLECTION_NAME, type='SEARCH',
    tags=[{'key': 'labrun:lab-slug', 'value': LAB_TAG_VALUE}],
)
collection_id = resp['createCollectionDetail']['id']
collection_detail = wait_collection_active(collection_id)
AOSS_HOST = collection_detail['collectionEndpoint']
```
For the index, call `os_client.indices.create(index=INDEX_NAME, body=_index_body)` and swallow `index_already_exists_exception`. The bulk-write loop iterates `CORPUS`, indexes a doc into OpenSearch, and does a parameterised pgvector upsert with the embedding from `embeddings_by_id`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
# Create collection
try:
    resp = aoss.create_collection(
        name=COLLECTION_NAME, type='SEARCH',
        tags=[{'key': 'labrun:lab-slug', 'value': LAB_TAG_VALUE}],
    )
    collection_id = resp['createCollectionDetail']['id']
except aoss.exceptions.ConflictException:
    collection_id = aoss.list_collections(
        collectionFilters={'name': COLLECTION_NAME}
    )['collectionSummaries'][0]['id']

collection_detail = wait_collection_active(collection_id)
AOSS_HOST = collection_detail['collectionEndpoint']

_host_no_scheme = AOSS_HOST.replace('https://', '').replace('http://', '')
os_client = OpenSearch(
    hosts=[{'host': _host_no_scheme, 'port': 443}],
    http_auth=_awsauth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, pool_maxsize=20,
)

try:
    os_client.indices.create(index=INDEX_NAME, body=_index_body)
except Exception as exc:
    if 'resource_already_exists' not in str(exc) and 'index_already_exists' not in str(exc):
        raise

conn = psycopg2.connect(DB_URI)
conn.autocommit = True
chunks_written = 0
with conn.cursor() as cur:
    for chunk_id, text, labels in CORPUS:
        os_client.index(
            index=INDEX_NAME,
            body={'chunk_id': chunk_id, 'text': text,
                  'topic': labels['topic'], 'subtopic': labels['subtopic']},
        )
        cur.execute(
            f'INSERT INTO public.{CHUNKS_TABLE} '
            '(chunk_id, text, topic, subtopic, embedding) '
            'VALUES (%s, %s, %s, %s, %s::vector) '
            'ON CONFLICT (chunk_id) DO UPDATE SET '
            '  text = EXCLUDED.text, embedding = EXCLUDED.embedding',
            (chunk_id, text, labels['topic'], labels['subtopic'],
             json.dumps(embeddings_by_id[chunk_id])),
        )
        chunks_written += 1
conn.close()

time.sleep(10)
os_client.indices.refresh(index=INDEX_NAME)
```

</details>

**Wallet check.** Retrieval substrate stood up. OpenSearch OCUs have been billing for about three minutes ($0.96/OCU-hr x 4 OCU x 0.05 hr = $0.19). OpenAI embeddings: 240 chunks at ~50 tokens each, ~12K tokens, $0.0002. Spent so far: about 20 cents. Your morning coffee cost 25x more.

## Task 2: Cost guardrail aborts a deliberately-expensive query

The CTO's first non-negotiable: no single query is allowed to cost more than 5 cents. The team got burned last quarter paying Cohere four hundred bucks for a runaway batch rerank, and that bug shipped because the cost estimator ran *after* the rerank, not *before*. You will write the guardrail so it estimates total cost from the request shape alone (token count, retrieval candidates, expected output tokens), aborts before any retrieval or rerank or generation runs, and returns the canonical abort string. A bypass flag (`bypass_guardrail=True`) on the same function proves the guardrail is the difference: with bypass, the expensive query completes normally; without bypass, the abort fires.

Pass criteria: when the expensive test query runs through `production_pipeline(query, bypass_guardrail=False)`, the response equals the canonical string `"query aborted: estimated cost $X exceeds 5 cent budget"` (X is the rounded estimate); the same query with `bypass_guardrail=True` returns a non-aborted response; the Phoenix trace logged for the aborted query has `aborted=true` on the guardrail span.

The cost guardrail is the cheapest line in the lab when it fires. The expensive query *did not run*. That is the point.

In [ ]:
# NBVAL_SKIP
# Level 2 skeleton. Trace emission helper, pricing constants, and the pipeline
# entry point are wired up. You write the cost estimator and the abort check.

# Pricing math (us-east-1 list price, early 2026):
#   OpenAI text-embedding-3-small: $0.02 per 1M tokens
#   Cohere Rerank 3.5: $1 per 1K calls (~$0.001 per call)
#   Claude Sonnet 4.5: $3 per 1M input + $15 per 1M output
EMBED_USD_PER_TOKEN = 0.02 / 1_000_000
RERANK_USD_PER_CALL = 0.001
SONNET_INPUT_USD_PER_TOKEN = 3.0 / 1_000_000
SONNET_OUTPUT_USD_PER_TOKEN = 15.0 / 1_000_000

# Lightweight Phoenix trace emitter. Failures are swallowed so traces never
# block the pipeline.
TRACE_BUFFER = []

def emit_trace(span_name, attrs):
    row = {
        "span": span_name,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "project": PHOENIX_PROJECT_NAME,
        "attrs": attrs,
    }
    TRACE_BUFFER.append(row)
    try:
        if OBS_PROVIDER == "phoenix" and PHOENIX_API_KEY:
            requests.post(
                "https://app.phoenix.arize.com/v1/spans",
                headers={"Authorization": f"Bearer {PHOENIX_API_KEY}"},
                json=row, timeout=5,
            )
        elif OBS_PROVIDER == "langfuse":
            requests.post(
                "https://cloud.langfuse.com/api/public/ingestion",
                auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
                json={"batch": [{"type": "span-create", "body": row}]},
                timeout=5,
            )
    except Exception:
        pass

def estimate_tokens(text):
    return max(1, int(len(text.split()) * 1.3))

def estimate_query_cost(query, retrieval_candidates=50, expected_output_tokens=200):
    """Return estimated query cost in dollars.

    Pre-retrieval estimate using:
      1) embedding cost = token_count * EMBED_USD_PER_TOKEN
      2) rerank cost    = RERANK_USD_PER_CALL (one rerank call per query)
      3) generation cost = (avg_chunk_tokens * top_k_after_rerank + query_tokens) * input_rate
                         + expected_output_tokens * output_rate
    The top_k_after_rerank is fixed at 5 for this lab's pipeline.
    """
    # YOUR CODE: compute embed_cost, rerank_cost, generation_cost in dollars,
    # sum them, and return the total.
    return 0.0

def guardrail_check(estimate_usd, budget_cents=COST_BUDGET_CENTS):
    """Return (allowed, abort_string). abort_string is None when allowed.

    Canonical abort string: 'query aborted: estimated cost $X exceeds 5 cent budget'
    where X is the rounded estimate to two decimal places.
    """
    # YOUR CODE: compare estimate_usd against budget_cents / 100.0.
    return (True, None)

_anthropic_client = anthropic_client
_co_client = co_client

@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, max=8))
def hybrid_retrieve(query, top_k=50):
    bm25 = os_client.search(
        index=INDEX_NAME,
        body={"size": top_k, "query": {"match": {"text": query}}},
    )
    bm25_hits = [(h["_source"]["chunk_id"], h["_score"]) for h in bm25["hits"]["hits"]]
    q_emb = embed_batch([query])[0]
    conn = psycopg2.connect(DB_URI)
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"SELECT chunk_id, text, 1 - (embedding <=> %s::vector) AS score "
                f"FROM public.{CHUNKS_TABLE} "
                f"ORDER BY embedding <=> %s::vector LIMIT %s",
                (json.dumps(q_emb), json.dumps(q_emb), top_k),
            )
            dense_hits = cur.fetchall()
    finally:
        conn.close()
    scores = {}
    texts = {}
    for cid, s in bm25_hits:
        scores[cid] = scores.get(cid, 0.0) + 0.5 * float(s)
    for cid, text, s in dense_hits:
        scores[cid] = scores.get(cid, 0.0) + 0.5 * float(s) * 5.0
        texts[cid] = text
    fetched = [(cid, scores[cid]) for cid in sorted(scores, key=scores.get, reverse=True)]
    return fetched[:top_k], texts

def rerank_chunks(query, candidates, texts, top_n=5):
    docs = [texts.get(cid, cid) for cid, _ in candidates]
    if not docs:
        return []
    resp = _co_client.rerank(
        model=COHERE_RERANK_MODEL, query=query, documents=docs, top_n=top_n,
    )
    return [(candidates[r.index][0], r.relevance_score) for r in resp.results]

def generate_answer(query, top_chunks, texts, system_prompt):
    context = "\n\n".join(
        f"[{cid}] {texts.get(cid, '')[:600]}" for cid, _ in top_chunks
    )
    resp = _anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=400,
        system=[{"type": "text", "text": system_prompt, "cache_control": {"type": "ephemeral"}}],
        messages=[{
            "role": "user",
            "content": f"Question: {query}\n\nContext:\n{context}\n\nAnswer:",
        }],
    )
    return resp.content[0].text

# The system prompt that holds the citation contract. Task 5 will sabotage
# this string. SYSTEM_PROMPT is module-level on purpose so the sabotage cell
# can rebind it.
SYSTEM_PROMPT = (
    "You are a production RAG assistant. Use only the supplied context. "
    "cite every chunk you reference using its [chunk_id] tag."
)

def production_pipeline(query, bypass_guardrail=False, system_prompt=None):
    sp = system_prompt if system_prompt is not None else SYSTEM_PROMPT
    estimate = estimate_query_cost(query)
    if not bypass_guardrail:
        allowed, abort_str = guardrail_check(estimate)
        if not allowed:
            emit_trace("guardrail", {"query": query[:120], "estimate_usd": estimate, "aborted": True})
            return {"response": abort_str, "aborted": True, "estimate_usd": estimate}
    emit_trace("guardrail", {"query": query[:120], "estimate_usd": estimate, "aborted": False})
    candidates, texts = hybrid_retrieve(query)
    top = rerank_chunks(query, candidates, texts)
    answer = generate_answer(query, top, texts, sp)
    emit_trace("answer", {"query": query[:120], "top_chunk_ids": [c for c, _ in top]})
    return {"response": answer, "aborted": False, "estimate_usd": estimate, "top_chunks": top}

# Deliberately-expensive test query. The padding is what blows the budget.
EXPENSIVE_QUERY = (
    "Explain in exhaustive detail every nuance of production RAG, including "
    "semantic caching, hybrid retrieval, reranking, prompt caching, cost "
    "guardrails, regression eval, chunk overlap, BM25 scoring, dense "
    "embeddings, recall at five, observability, monitoring, common pitfalls, "
    "failure modes, cost math, tuning levers, production patterns, and "
    "trade-offs. "
) * 80

guarded = production_pipeline(EXPENSIVE_QUERY, bypass_guardrail=False)
print("Guarded response:", guarded["response"][:120])

# Bypass run on a SHORT query: we only confirm the guardrail is the difference,
# we do not actually let the expensive call complete.
_short = "What is production rag?"
bypassed = production_pipeline(_short, bypass_guardrail=True)
print("Bypassed response (short query, proves the non-guarded path runs):", bypassed["response"][:120])

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: canonical abort string returned for the expensive query;
# non-guarded path returns a normal response on a short query; trace buffer
# contains an aborted=True row.

CANONICAL_ABORT_RE = re.compile(
    r"^query aborted: estimated cost \$\d+\.\d{2} exceeds 5 cent budget$"
)

def checkpoint_2(session):
    if not isinstance(guarded, dict) or "response" not in guarded:
        return CheckpointResult(
            status="fail",
            error_reason="production_pipeline did not return a dict with a 'response' key for the expensive query.",
        )
    resp = guarded.get("response", "")
    if not CANONICAL_ABORT_RE.match((resp or "").strip()):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Guarded response did not match the canonical abort string. "
                f"Got: {resp[:200]!r}. Expected pattern: "
                f"'query aborted: estimated cost $X.XX exceeds 5 cent budget'. "
                f"Confirm guardrail_check returns the literal canonical string when over budget."
            ),
        )
    if not guarded.get("aborted"):
        return CheckpointResult(
            status="fail",
            error_reason="Guarded response is the abort string but aborted=False; the pipeline kept running after the abort.",
        )
    if not isinstance(bypassed, dict) or bypassed.get("aborted"):
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Bypassed run was either not a dict or had aborted=True. The bypass flag "
                "must skip guardrail_check entirely so the non-guarded path runs."
            ),
        )
    aborted_trace = next((t for t in TRACE_BUFFER if t["span"] == "guardrail" and t["attrs"].get("aborted") is True), None)
    if aborted_trace is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "No Phoenix/Langfuse trace span found with aborted=True. Confirm emit_trace fires "
                "inside production_pipeline on the abort branch."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two functions need bodies: `estimate_query_cost` and `guardrail_check`. The cost is the sum of three pre-retrieval projections (embed, rerank, generation). The guardrail returns the canonical string verbatim when the estimate is over budget. The abort path MUST run before any retrieval or rerank call.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def estimate_query_cost(query, retrieval_candidates=50, expected_output_tokens=200):
    q_tokens = estimate_tokens(query)
    embed_cost = q_tokens * EMBED_USD_PER_TOKEN
    rerank_cost = RERANK_USD_PER_CALL
    projected_input = q_tokens + 5 * 150
    generation_cost = projected_input * SONNET_INPUT_USD_PER_TOKEN + expected_output_tokens * SONNET_OUTPUT_USD_PER_TOKEN
    return embed_cost + rerank_cost + generation_cost
```
For `guardrail_check`, compare against `budget_cents / 100.0` and return the canonical string with the estimate rounded to two decimal places.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
def estimate_query_cost(query, retrieval_candidates=50, expected_output_tokens=200):
    q_tokens = estimate_tokens(query)
    embed_cost = q_tokens * EMBED_USD_PER_TOKEN
    rerank_cost = RERANK_USD_PER_CALL
    projected_input_tokens = q_tokens + 5 * 150
    generation_cost = (
        projected_input_tokens * SONNET_INPUT_USD_PER_TOKEN
        + expected_output_tokens * SONNET_OUTPUT_USD_PER_TOKEN
    )
    return embed_cost + rerank_cost + generation_cost

def guardrail_check(estimate_usd, budget_cents=COST_BUDGET_CENTS):
    budget_usd = budget_cents / 100.0
    if estimate_usd <= budget_usd:
        return (True, None)
    rounded = round(estimate_usd, 2)
    return (False, f'query aborted: estimated cost ${rounded:.2f} exceeds 5 cent budget')
```

</details>

**Wallet check.** Guardrail just saved you about 7 cents (the expensive query did not run; the abort string returned before any retrieval, rerank, or generation). The bypassed short-query path cost a fraction of a cent. Spent so far: about 22 cents. OpenSearch OCUs keep ticking in the background regardless; that is the biggest line in the lab and you cannot guardrail it.

## Task 3: Semantic cache hits at least 30 percent on a deterministic test set

The CTO's second non-negotiable: most production RAG traffic is a long tail of repeat queries ("how do I reset my password?"). The bot should hit a cache, not the full retrieval stack, when it sees a semantic duplicate. The eval set from the corpus seed has 100 queries; 30 of them are deliberate paraphrases of the first 30 originals (cosine similarity of the embeddings should land around 0.95 to 0.97). Your job: wire up the semantic cache so it returns a cached response when the incoming query's embedding matches a previously-cached embedding within `cosine >= 0.95`, and the cache write happens *before* the response is returned (not after, because the next paraphrase arrives within milliseconds in production traffic).

The cache lives in Upstash Redis. Each entry is keyed by `{UPSTASH_CACHE_PREFIX}{embedding_hash}` (SHA-256 of the embedding bytes for a fast key) and stores a small JSON blob with the original query, the response, and the embedding itself for similarity comparison. Hit counts and miss counts go in two dedicated counter keys so the validator can read them.

Pass criteria: after running all 100 eval queries through `cached_pipeline()`, the Upstash hit counter is at least 30. Loosen the threshold (the tier 2 hint shows how) only if you are confident the paraphrase distribution is tighter than the default.

In [ ]:
# NBVAL_SKIP
# Level 2 skeleton. Cache key derivation, hit/miss counter increment, and
# cached_pipeline wiring are provided. You write the cache lookup, the cache
# write, and the similarity check.

upstash.delete(UPSTASH_HIT_COUNTER_KEY, UPSTASH_MISS_COUNTER_KEY)

def _l2(v):
    s = math.sqrt(sum(x * x for x in v))
    return [x / s for x in v] if s > 0 else v

def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b))

def cache_lookup(query_embedding, threshold=CACHE_SIMILARITY_THRESHOLD):
    """Return cached response dict if a cached embedding matches within threshold, else None.

    Steps:
      1) SCAN Upstash keys with prefix UPSTASH_CACHE_PREFIX
      2) For each, GET the JSON blob, compare cosine on the stored embedding
      3) Return the first match where cosine >= threshold
    """
    # YOUR CODE: implement the scan + compare loop.
    return None

def cache_write(query, response, query_embedding):
    """Store the (query, response, embedding) tuple in Upstash before returning."""
    # YOUR CODE: compute a key from the embedding, serialise the blob as JSON,
    # and upstash.set the key.
    pass

def cached_pipeline(query):
    q_emb = _l2(embed_batch([query])[0])
    cached = cache_lookup(q_emb)
    if cached is not None:
        upstash.incr(UPSTASH_HIT_COUNTER_KEY)
        emit_trace("cache", {"query": query[:120], "cache_hit": True})
        return {"response": cached["response"], "from_cache": True, "top_chunks": cached.get("top_chunks", [])}
    result = production_pipeline(query)
    if result.get("aborted"):
        upstash.incr(UPSTASH_MISS_COUNTER_KEY)
        emit_trace("cache", {"query": query[:120], "cache_hit": False, "aborted": True})
        return result
    # CACHE WRITE BEFORE RETURN. The next paraphrase arrives in milliseconds in
    # production traffic; writing after the response means the second query
    # sees a cold cache.
    cache_write(query=query, response=result["response"], query_embedding=q_emb)
    upstash.incr(UPSTASH_MISS_COUNTER_KEY)
    emit_trace("cache", {"query": query[:120], "cache_hit": False})
    return {"response": result["response"], "from_cache": False, "top_chunks": [c for c, _ in result.get("top_chunks", [])]}

print("Running 100 queries through the cached pipeline...")
messages = [
    "first pass, cache should be cold",
    "paraphrases coming up",
    "still going, the cache is doing its job",
    "almost done",
]
_t0 = time.time()
for i, q_meta in enumerate(EVAL_SET):
    cached_pipeline(q_meta["query"])
    if i % 25 == 0:
        print(f"  {messages[(i // 25) % len(messages)]} ({i + 1}/{len(EVAL_SET)})")
print(f"100 queries complete in {int(time.time() - _t0)}s.")
hit_count = int(upstash.get(UPSTASH_HIT_COUNTER_KEY) or 0)
miss_count = int(upstash.get(UPSTASH_MISS_COUNTER_KEY) or 0)
print(f"Cache hits: {hit_count} / {hit_count + miss_count}. Hit rate: {100 * hit_count / max(1, hit_count + miss_count):.1f}%.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Upstash hit counter >= 30; cache_hit=True traces present for
# at least 30 queries.

def checkpoint_3(session):
    try:
        hits = int(upstash.get(UPSTASH_HIT_COUNTER_KEY) or 0)
        misses = int(upstash.get(UPSTASH_MISS_COUNTER_KEY) or 0)
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Upstash counter read failed: {exc!r}",
        )
    if hits < 30:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Cache hit count {hits} is below 30 (total queries seen: {hits + misses}). "
                f"Most common cause: cache_write was called AFTER the response was returned, so "
                f"the second paraphrase saw a cold cache. Confirm cache_write fires before "
                f"cached_pipeline returns. Second most common cause: the similarity threshold is "
                f"too strict; default is {CACHE_SIMILARITY_THRESHOLD}, loosening to 0.92 typically "
                f"lifts hit rate by 5 to 10 points."
            ),
        )
    trace_hits = sum(1 for t in TRACE_BUFFER if t["span"] == "cache" and t["attrs"].get("cache_hit") is True)
    if trace_hits < 30:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Trace span cache_hit=True count is {trace_hits}, expected >= 30. "
                f"emit_trace is firing on the wrong branch or not at all."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two functions: `cache_lookup` and `cache_write`. The lookup iterates Upstash keys with the right prefix, deserialises each, and returns the first one whose stored embedding matches the incoming embedding above the threshold. The write stores a JSON blob keyed by something deterministic from the embedding. Both are small loops; the trick is making sure `cache_write` lands BEFORE the response is returned so the next paraphrase hits.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def cache_lookup(query_embedding, threshold=CACHE_SIMILARITY_THRESHOLD):
    cursor = 0
    while True:
        cursor, batch = upstash.scan(cursor, match=f'{UPSTASH_CACHE_PREFIX}*', count=200)
        for key in batch:
            raw = upstash.get(key)
            if not raw: continue
            blob = json.loads(raw)
            if cosine_similarity(query_embedding, blob['embedding']) >= threshold:
                return blob
        if cursor == 0: break
    return None
```
For the write, derive the key from `hashlib.sha256(json.dumps(query_embedding).encode()).hexdigest()[:16]` and store the JSON blob via `upstash.set(key, json.dumps(...))`. If your hit rate is below 30, loosen `CACHE_SIMILARITY_THRESHOLD` from 0.95 to 0.92 (the paraphrase distribution in the synthetic corpus sits right around 0.93 to 0.96).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
def cache_lookup(query_embedding, threshold=CACHE_SIMILARITY_THRESHOLD):
    cursor = 0
    while True:
        cursor, batch = upstash.scan(
            cursor, match=f'{UPSTASH_CACHE_PREFIX}*', count=200,
        )
        for key in batch:
            raw = upstash.get(key)
            if not raw:
                continue
            try:
                blob = json.loads(raw)
            except Exception:
                continue
            cached_emb = blob.get('embedding')
            if not cached_emb:
                continue
            sim = cosine_similarity(query_embedding, cached_emb)
            if sim >= threshold:
                return blob
        if cursor == 0:
            break
    return None

def cache_write(query, response, query_embedding):
    key_seed = json.dumps(query_embedding).encode()
    key = UPSTASH_CACHE_PREFIX + hashlib.sha256(key_seed).hexdigest()[:16]
    blob = json.dumps({
        'query': query,
        'response': response,
        'embedding': query_embedding,
    })
    upstash.set(key, blob)

# If the hit rate is below 30, loosen the threshold:
CACHE_SIMILARITY_THRESHOLD = 0.92
```

</details>

**Wallet check.** Cache hit rate around 30 percent on a 100-query set means you skipped 30 full retrieval-plus-rerank-plus-generation cycles. Each skipped cycle saves about half a cent in Cohere + Claude calls. Roughly 15 cents saved by the cache alone. Spent so far: about $1.10. OpenSearch OCUs continue to be the dominant line.

## Task 4: Regression eval baseline against the 100-query golden set

The CTO's third non-negotiable: a regression eval suite that blocks deploys when recall@5 drops more than 3 points. To detect regressions you need a baseline; this task runs the current production prompt against the 100-query golden set, computes recall@5 per query, and writes the results to two Supabase tables. One row per query in `labrun_production_rag_cache_eval_eval_results`, one summary row in `labrun_production_rag_cache_eval_regression_summary`. All timestamps in UTC. The summary recall@5 should land at or above 0.80 (the current production prompt is good; the synthetic corpus is structured so the citation contract produces strong recall).

Pass criteria: 100 per-query rows + 1 summary row from this run timestamp, summary `recall_at_5 >= 0.80`. The validator queries Supabase for the row counts and reads the summary; do not be tempted to compute the summary in memory and skip the write.

Use UTC timestamps. The regression detector in Task 5 orders runs by `run_timestamp DESC`; local-time timestamps mis-sort across DST boundaries and across users in different timezones.

In [ ]:
# NBVAL_SKIP
# Level 2 skeleton. Table DDL, the eval-loop entry point, and the summary
# row write are wired up. You write the per-query recall@5 computation and
# the per-query INSERT.

_eval_results_ddl = f"""
CREATE TABLE IF NOT EXISTS public.{EVAL_RESULTS_TABLE} (
    id bigserial PRIMARY KEY,
    run_id text NOT NULL,
    run_label text NOT NULL,
    query_id text NOT NULL,
    query text NOT NULL,
    recall_at_5 double precision,
    top_chunks jsonb,
    run_timestamp timestamptz NOT NULL DEFAULT now()
);
"""
_regression_summary_ddl = f"""
CREATE TABLE IF NOT EXISTS public.{REGRESSION_SUMMARY_TABLE} (
    id bigserial PRIMARY KEY,
    run_id text NOT NULL UNIQUE,
    run_label text NOT NULL,
    recall_at_5 double precision NOT NULL,
    query_count integer NOT NULL,
    run_timestamp timestamptz NOT NULL DEFAULT now()
);
"""
_conn = psycopg2.connect(DB_URI)
_conn.autocommit = True
with _conn.cursor() as _cur:
    _cur.execute(_eval_results_ddl)
    _cur.execute(_regression_summary_ddl)
_conn.close()
print("Eval tables ready.")

def recall_at_5(predicted_chunk_ids, gold_chunk_ids):
    """Fraction of gold chunks that appear in the top-5 predicted chunks."""
    # YOUR CODE: compute |predicted_top_5 \cap gold| / |gold|; handle empty gold.
    return 0.0

def run_eval_suite(run_label):
    run_id = f"{RUN_ID_PREFIX}-{run_label}-{int(time.time())}"
    ts = datetime.now(timezone.utc)
    conn = psycopg2.connect(DB_URI)
    conn.autocommit = True
    recalls = []
    completed = 0
    try:
        with conn.cursor() as cur:
            for q_meta in EVAL_SET:
                try:
                    pipe_result = cached_pipeline(q_meta["query"])
                    if pipe_result.get("aborted"):
                        r = float("nan")
                        top_chunks = []
                    else:
                        top_chunks_raw = pipe_result.get("top_chunks", [])
                        if top_chunks_raw and isinstance(top_chunks_raw[0], (list, tuple)):
                            top_chunks = [c for c, _ in top_chunks_raw]
                        else:
                            top_chunks = list(top_chunks_raw)
                        r = recall_at_5(top_chunks, q_meta["gold_chunk_ids"])
                except Exception as exc:
                    r = float("nan")
                    top_chunks = []
                    print(f"  warn: query {q_meta['query_id']} crashed: {exc!r}")
                # YOUR CODE: INSERT one row into EVAL_RESULTS_TABLE with
                # (run_id, run_label, query_id, query, recall_at_5, top_chunks, run_timestamp).
                # Use the cur cursor and the ts timestamp.
                recalls.append(r if not math.isnan(r) else 0.0)
                completed += 1
            summary = sum(recalls) / max(1, len(recalls))
            cur.execute(
                f"INSERT INTO public.{REGRESSION_SUMMARY_TABLE} "
                "(run_id, run_label, recall_at_5, query_count, run_timestamp) "
                "VALUES (%s, %s, %s, %s, %s)",
                (run_id, run_label, summary, completed, ts),
            )
    finally:
        conn.close()
    snapshot_path = f"/content/regression_run_{run_label}.json"
    with open(snapshot_path, "w") as f:
        json.dump({
            "run_id": run_id, "run_label": run_label,
            "recall_at_5": summary, "timestamp": ts.isoformat(),
            "per_query": recalls,
        }, f, indent=2)
    emit_trace("eval_run", {"run_id": run_id, "run_label": run_label, "recall_at_5": summary})
    print(f"Eval run {run_label}: recall@5 = {summary:.3f} across {completed} queries.")
    return run_id, summary

baseline_run_id, baseline_recall = run_eval_suite("baseline")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: 100 per-query rows + 1 summary row written; summary recall >= 0.80;
# run_timestamp is UTC.

def checkpoint_4(session):
    try:
        conn = psycopg2.connect(DB_URI)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    f"SELECT COUNT(*) FROM public.{EVAL_RESULTS_TABLE} WHERE run_id = %s",
                    (baseline_run_id,),
                )
                per_query_count = cur.fetchone()[0]
                cur.execute(
                    f"SELECT recall_at_5, query_count, run_timestamp "
                    f"FROM public.{REGRESSION_SUMMARY_TABLE} WHERE run_id = %s",
                    (baseline_run_id,),
                )
                summary_row = cur.fetchone()
        finally:
            conn.close()
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Supabase read failed: {exc!r}",
        )
    if per_query_count != 100:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Expected 100 per-query rows for run_id={baseline_run_id!r}, found {per_query_count}. "
                f"Partial completion usually means the eval loop crashed mid-way (often a 429 from "
                f"one of the providers). Confirm the per-query INSERT inside run_eval_suite fires for "
                f"every query, not only the successful ones."
            ),
        )
    if summary_row is None:
        return CheckpointResult(
            status="fail",
            error_reason=f"No summary row found for run_id={baseline_run_id!r}.",
        )
    summary_recall, qc, run_ts = summary_row
    if summary_recall is None or summary_recall < 0.80:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Summary recall@5 = {summary_recall}, expected >= 0.80. Most common cause: "
                f"chunk-ID mismatch between the eval labels and the indexed corpus. Confirm the "
                f"`gold_chunk_ids` in EVAL_SET reference chunk_ids that exist in the OpenSearch index."
            ),
        )
    if run_ts is None or run_ts.tzinfo is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "run_timestamp is timezone-naive. Use datetime.now(timezone.utc) when inserting; the "
                "regression detector orders runs by UTC timestamp."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two pieces of code: the `recall_at_5` body and the per-query INSERT in `run_eval_suite`. Recall is the count of gold chunks that appear in the top-5 predicted chunks divided by the count of gold chunks. The INSERT lands one row per query with the UTC timestamp; the summary row is already wired up at the bottom of the loop.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def recall_at_5(predicted_chunk_ids, gold_chunk_ids):
    if not gold_chunk_ids:
        return 0.0
    top5 = set(predicted_chunk_ids[:5])
    hits = len(top5 & set(gold_chunk_ids))
    return hits / len(gold_chunk_ids)

# Inside the loop, after computing r and top_chunks:
cur.execute(
    f'INSERT INTO public.{EVAL_RESULTS_TABLE} '
    '(run_id, run_label, query_id, query, recall_at_5, top_chunks, run_timestamp) '
    'VALUES (%s, %s, %s, %s, %s, %s::jsonb, %s)',
    (run_id, run_label, q_meta['query_id'], q_meta['query'],
     r if not math.isnan(r) else None, json.dumps(top_chunks), ts),
)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
def recall_at_5(predicted_chunk_ids, gold_chunk_ids):
    if not gold_chunk_ids:
        return 0.0
    top5 = set(predicted_chunk_ids[:5])
    gold = set(gold_chunk_ids)
    hits = len(top5 & gold)
    return hits / len(gold)

# Per-query INSERT (inside the for-loop, right after r and top_chunks):
cur.execute(
    f'INSERT INTO public.{EVAL_RESULTS_TABLE} '
    '(run_id, run_label, query_id, query, recall_at_5, top_chunks, run_timestamp) '
    'VALUES (%s, %s, %s, %s, %s, %s::jsonb, %s)',
    (
        run_id, run_label, q_meta['query_id'], q_meta['query'],
        r if not math.isnan(r) else None,
        json.dumps(top_chunks),
        ts,
    ),
)
```

</details>

**Wallet check.** Baseline eval run cost: 100 queries through hybrid retrieve + rerank + generation, minus the 30 or so cache hits. Roughly $0.50 in Claude + $0.10 in Cohere + $0.01 in OpenAI embeddings. Spent so far: about $1.75. OpenSearch OCUs are still the biggest line. Anthropic prompt caching is doing real work; same system prompt re-used across all 100 calls means the cached prefix hits at 10 percent of the input price.

## Task 5: The killer moment, regression eval catches the sabotaged prompt and blocks the deploy

You have a baseline. The CTO has been burned twice in six months by silent regressions in the system prompt; the regression detector is the guardrail that catches the next one. We are going to deliberately ship one. The sabotage cell below mutates `SYSTEM_PROMPT` in a way that looks innocent in a code review ("be concise" sounds fine) but drops the citation contract. The re-run eval suite recomputes recall@5 against the same 100-query golden set. The regression detector reads the two most recent rows from `regression_summary` ordered by `run_timestamp DESC` and returns `should_block_deploy=True` when `prior.recall_at_5 - current.recall_at_5 >= 0.03`.

Pass criteria: after running the sabotage cell and re-running the eval suite, the regression detector's verdict object reports `should_block_deploy=True`, the recorded drop is at least 3 points, and the two most recent runs in `regression_summary` are the baseline (from Task 4) and the sabotaged run (from this task).

This is the moment the lab is built around. You watched your own regression detector save you from shipping a broken prompt.

In [ ]:
# NBVAL_SKIP
# Sabotage cell. We mutate SYSTEM_PROMPT in place. The citation contract is
# gone; the bot now prefers concision over citing chunks, which drops recall@5
# by 8 to 12 points on the synthetic corpus.

print("Before sabotage, SYSTEM_PROMPT contained the citation contract:")
print(" ", SYSTEM_PROMPT[:140])

SYSTEM_PROMPT = SYSTEM_PROMPT.replace(
    "cite every chunk you reference using its [chunk_id] tag.",
    "be concise.",
)

print()
print("After sabotage, the citation contract is gone:")
print(" ", SYSTEM_PROMPT[:140])

# Clear the semantic cache so the sabotaged prompt runs on every query (a
# cached hit from the baseline run would mask the regression).
cursor = 0
deleted_keys = 0
while True:
    cursor, batch = upstash.scan(cursor, match=f"{UPSTASH_CACHE_PREFIX}*", count=200)
    if batch:
        upstash.delete(*batch)
        deleted_keys += len(batch)
    if cursor == 0:
        break
upstash.delete(UPSTASH_HIT_COUNTER_KEY, UPSTASH_MISS_COUNTER_KEY)
print(f"Cleared {deleted_keys} cache keys. Sabotaged prompt will run on every query.")

sabotaged_run_id, sabotaged_recall = run_eval_suite("sabotaged")
print(f"Baseline recall: {baseline_recall:.3f}. Sabotaged recall: {sabotaged_recall:.3f}.")

In [ ]:
# NBVAL_SKIP
# Regression detector. Reads the two most recent rows from regression_summary
# ordered by run_timestamp DESC. Returns a verdict object with
# should_block_deploy, prior_recall, current_recall, and delta.

def detect_regression(threshold=REGRESSION_DELTA_THRESHOLD):
    """Compare the most recent run to the immediately-prior run.

    Returns a dict:
      {
        'should_block_deploy': bool,
        'prior_run_id': str, 'current_run_id': str,
        'prior_recall': float, 'current_recall': float,
        'delta': float,         # prior - current
        'threshold': float,
      }
    """
    # YOUR CODE: SELECT run_id, recall_at_5, run_timestamp FROM regression_summary
    # ORDER BY run_timestamp DESC LIMIT 2; rows[0] is current, rows[1] is prior;
    # delta = prior_recall - current_recall; should_block_deploy = delta >= threshold.
    return {
        "should_block_deploy": False,
        "prior_run_id": None,
        "current_run_id": None,
        "prior_recall": None,
        "current_recall": None,
        "delta": 0.0,
        "threshold": threshold,
    }

verdict = detect_regression()
emit_trace("regression_detector", verdict)
print("Verdict:")
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: verdict.should_block_deploy is True; delta >= 0.03; verdict
# uses the two most recent runs by run_timestamp DESC; trace span exists.

def checkpoint_5(session):
    if not isinstance(verdict, dict):
        return CheckpointResult(
            status="fail",
            error_reason="detect_regression did not return a dict.",
        )
    required = {"should_block_deploy", "prior_run_id", "current_run_id", "prior_recall", "current_recall", "delta"}
    missing = required - set(verdict.keys())
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=f"Verdict object missing keys: {sorted(missing)}.",
        )
    if verdict.get("current_run_id") != sabotaged_run_id:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"current_run_id is {verdict.get('current_run_id')!r}; expected the sabotaged run "
                f"{sabotaged_run_id!r}. Confirm you ORDER BY run_timestamp DESC and take LIMIT 2, "
                f"with the FIRST row as current."
            ),
        )
    if verdict.get("prior_run_id") != baseline_run_id:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"prior_run_id is {verdict.get('prior_run_id')!r}; expected the baseline run "
                f"{baseline_run_id!r}. The detector should read exactly the immediately-prior run, "
                f"not a median or an older row."
            ),
        )
    delta = verdict.get("delta") or 0.0
    if delta < REGRESSION_DELTA_THRESHOLD:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Delta is {delta:.4f}, below threshold {REGRESSION_DELTA_THRESHOLD}. The sabotage "
                f"prompt is supposed to drop recall by 8 to 12 points. If the drop is smaller, the "
                f"cache was not cleared before the sabotaged run, so cached hits from the baseline "
                f"masked the regression."
            ),
        )
    if not verdict.get("should_block_deploy"):
        return CheckpointResult(
            status="fail",
            error_reason=(
                "should_block_deploy is False but delta is above the threshold. The verdict logic "
                "is inverted. Block when delta >= threshold, allow otherwise."
            ),
        )
    trace_hit = any(t["span"] == "regression_detector" for t in TRACE_BUFFER)
    if not trace_hit:
        return CheckpointResult(
            status="fail",
            error_reason="No regression_detector trace span recorded. Call emit_trace after detect_regression.",
        )
    return CheckpointResult(status="pass")


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

The detector reads the two most recent rows from `regression_summary` ordered by `run_timestamp DESC`. The first row is the current run (sabotaged), the second is the prior (baseline). The delta is `prior_recall - current_recall`. Block when `delta >= 0.03`. If your verdict object has `prior` and `current` swapped, the delta is negative and the block never fires.

</details>

<details><summary>Hint 2 (direction)</summary>

```
conn = psycopg2.connect(DB_URI)
with conn.cursor() as cur:
    cur.execute(
        f'SELECT run_id, recall_at_5, run_timestamp '
        f'FROM public.{REGRESSION_SUMMARY_TABLE} '
        f'ORDER BY run_timestamp DESC LIMIT 2',
    )
    rows = cur.fetchall()
conn.close()
current, prior = rows[0], rows[1]
delta = prior[1] - current[1]
should_block = delta >= threshold
```
Then assemble the verdict dict with `current_run_id=current[0]`, `prior_run_id=prior[0]`, etc.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5:

```python
def detect_regression(threshold=REGRESSION_DELTA_THRESHOLD):
    conn = psycopg2.connect(DB_URI)
    try:
        with conn.cursor() as cur:
            cur.execute(
                f'SELECT run_id, recall_at_5, run_timestamp '
                f'FROM public.{REGRESSION_SUMMARY_TABLE} '
                f'ORDER BY run_timestamp DESC LIMIT 2'
            )
            rows = cur.fetchall()
    finally:
        conn.close()
    if len(rows) < 2:
        return {
            'should_block_deploy': False,
            'prior_run_id': None,
            'current_run_id': rows[0][0] if rows else None,
            'prior_recall': None,
            'current_recall': rows[0][1] if rows else None,
            'delta': 0.0,
            'threshold': threshold,
            'reason': 'fewer than two runs available',
        }
    current = rows[0]
    prior = rows[1]
    delta = float(prior[1]) - float(current[1])
    return {
        'should_block_deploy': bool(delta >= threshold),
        'prior_run_id': prior[0],
        'current_run_id': current[0],
        'prior_recall': float(prior[1]),
        'current_recall': float(current[1]),
        'delta': delta,
        'threshold': threshold,
    }
```

</details>

**Wallet check.** Regression detector just saved you from shipping a bad prompt. The sabotaged prompt dropped recall@5 by 8 to 12 points; the detector returned `should_block_deploy=True` and the deploy is blocked. In production this verdict would gate the CD pipeline; the deploy never reaches users. Total session damage so far: about $2.50, mostly OpenSearch OCUs.

## Cleanup

Critical-first per RESOURCE-SAFETY-SPEC Section 2 Rule 2. The OpenSearch Serverless collection comes down first (largest hourly bill); then the three security policies; then the EventBridge rule + Lambda + IAM role (the two-hour watchdog); then Upstash keys; then the Phoenix project; then the three pgvector tables; then the local regression snapshots. The adapter swallows already-gone errors per surface and re-raises everything else so `run_cleanup` turns the failure into a warning with the exact CLI command.

In [ ]:
# NBVAL_SKIP
# Final cleanup. Wraps run_cleanup, prints the canonical RESOURCE-SAFETY-SPEC
# Rule 6 summary, and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = sum(1 for r in CLEANUP_MANIFEST if not r.critical)
critical_failed = sum(1 for r in CLEANUP_MANIFEST if r.critical and r.id in failed_ids)
standard_failed = sum(1 for r in CLEANUP_MANIFEST if not r.critical and r.id in failed_ids)
critical_destroyed = critical_total - critical_failed
standard_destroyed = standard_total - standard_failed
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
if failed_count > 0:
    print("If K > 0, your AWS account or Supabase project may still incur charges. Resolve before closing this notebook.")
else:
    print("All clear. Every resource this lab created has been verified deleted.")
    print("OpenSearch OCUs stopped billing when the collection went DELETED.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $2.50 to $3.00.** OpenSearch Serverless OCUs across a 90 to 120 minute session were the biggest line at roughly $1.90. Claude Sonnet across the eval runs landed around $0.40 (prompt caching saved roughly 90 percent on the citation contract prefix). Cohere rerank across 200ish calls cost around $0.20. OpenAI embeddings for the corpus plus the 200ish eval queries cost roughly $0.02. Upstash, Phoenix, Supabase, IAM, Lambda, and EventBridge are all on free tier or near-zero. Cleanup just dropped the collection and the OCUs stopped billing the moment the DELETED state landed; check the AWS Billing console in 24 hours to confirm the exact number.

## Reflection

These are not graded. They are for you.

1. The cost guardrail aborts queries above 5 cents. In production the team is considering moving the threshold to a per-tenant configurable value (some enterprise tenants are willing to pay more per query for premium SLAs). What is the simplest data shape that supports per-tenant thresholds without changing the guardrail function's signature? Walk through the trade-off: tenant_id-keyed dict vs database lookup vs config-file-driven. What is one downside of the simplest option?

2. The semantic cache uses a cosine 0.95 threshold. Imagine a worst-case false-positive: two queries that match at 0.97 but actually have different intent ("reset my password" vs "reset my data"). What is one mitigation that does NOT involve tightening the threshold? Hint: think about what additional features beyond the embedding could disambiguate intent.

3. The regression detector reads the two most recent runs. In a real CD pipeline the eval suite runs on every PR, which means dozens of runs per day. What is one failure mode of the compare-to-immediately-prior approach when runs are dense, and what is one robustness improvement?

## Exam-Style Practice

**Question 1 (regression eval semantics):**

A production RAG service has a regression eval suite that blocks deploys when recall@5 drops more than 3 points. After a routine model upgrade, the suite reports a 4-point drop on a 100-query eval set. The eng lead asks: should we ship anyway with a feature flag, or block the deploy entirely?

A. Block the deploy entirely; 4 points is above the threshold and the threshold is the contract.
B. Ship behind a feature flag with 1% traffic and re-evaluate against production traffic.
C. Re-run the eval suite 5 more times and take the median; if median is below threshold, ship.
D. Roll back to the prior model version and investigate the regression in a separate branch.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong as a blanket answer. The threshold is a default; treating it as immutable removes engineering judgment. The eng lead is asking the right question.
- B is correct. A 4-point drop on a 100-query eval set has a non-trivial confidence interval; a 1% production rollout captures real-traffic signal without exposing all users. The regression detector keeps monitoring.
- C is wrong. Re-running the same deterministic eval gives the same result; it gathers no new information.
- D is defensible in some contexts (investigate before ship is the older pattern) but B is the modern ML team approach: ship-and-learn behind a flag rather than block-and-stall.

</details>

**Question 2 (semantic cache correctness):**

A semantic cache returns a cached response when the cosine similarity between the new query's embedding and a cached query's embedding is at least 0.95. A user complains that asking "reset my password" returns a response originally cached for "reset my data". The two embeddings have cosine 0.97. Which mitigation is most appropriate?

A. Tighten the similarity threshold to 0.99 globally.
B. Add an intent classifier as a secondary check; both queries must share predicted intent before a cache hit returns.
C. Disable the semantic cache entirely.
D. Use a larger embedding model so the cosine similarity drops below 0.95 for these two queries.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong. Tightening to 0.99 destroys cache hit rate across the long tail of legitimate paraphrases; the cure is worse than the disease.
- B is correct. A secondary intent check disambiguates queries with high embedding similarity but different intent; the cache stays effective for true paraphrases.
- C is wrong. Disabling the cache forfeits the 30%+ cost savings; the business case for the cache stays intact.
- D is wrong. A larger embedding model is expensive and slow; cosine similarity for these specific queries may not drop below 0.95 anyway because the two queries are textually almost identical.

</details>

**Question 3 (cost guardrail placement):**

A team's RAG pipeline computes cost AFTER retrieval and rerank, then aborts the query if the cost is above budget. The guardrail logs `aborted=true` on the trace. A senior engineer reviews the trace and says the guardrail is placed wrong. Why?

A. The cost should be logged in dollars, not cents.
B. The guardrail should run BEFORE retrieval and rerank; running it after means the team has already paid for the most expensive line items by the time the abort fires.
C. The Phoenix trace should use a different attribute name.
D. The guardrail should be implemented in the LLM provider, not the application layer.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong. Units are a stylistic choice; the placement is the real issue.
- B is correct. The whole point of a cost guardrail is to prevent expensive operations from running. If the estimator runs after retrieval and rerank (the most expensive operations), the team has already paid for them by the time the abort triggers. The estimator must use the request shape alone (token count, retrieval candidates, expected output tokens) and run before any paid call.
- C is wrong. Trace attribute naming does not change the cost model.
- D is wrong. LLM providers do not know what your retrieval and rerank costs are; the guardrail belongs in the application layer where the full cost picture is visible.

</details>